# 03h — 2단계 학습률 (STEP 17)

## 묻는 것 하나

STEP 16 에서 2단계(`convnextv2_base`, 89M)의 **best 가 0에폭**이었습니다.
14에폭까지 돌려도 못 넘었고, 그 사이 **train loss 가 올라갔습니다.**

```
ep0   train 1.0292  val 1.2549  macroF1 0.5973 ★best
ep2   train 1.2321  val 1.2766  0.5728    ← train loss 꼭대기
ep10  train 0.6922  val 1.4325  0.5914    ← 꺾임
```

설명이 둘입니다:

| | 가설 | 맞다면 |
|---|---|---|
| ① | warmup 2에폭이라 원래 그렇다 | 지금 0.599 가 이 레시피의 실력 |
| ② | **학습률이 높다** — 백본 9e-5 를 7,569스텝/에폭 | 0.599 는 천장이 아님 |

②가 맞으면 **A4 recall 0.264 도 배율 하락 26.8% 도 "덜 배운 모델에서 잰 값"** 이
됩니다. STEP 9→12 에서 이미 같은 일을 당했습니다 — 미수렴 기준선이 교란 검사를
왜곡했고, resnet50 을 수렴시키니 배율 하락이 12.2% → 15.1% 로 **나빠졌습니다.**

## ★ 판정 기준 — 돌리기 전에 못 박습니다

`src/experiments.py` 의 `lr_report()` 에 코드로 들어 있습니다 (노트북 셀은
`git pull` 로 안 바뀌므로 판정은 `src/` 에 둡니다 — 작업 규칙 3).

| | 기준 | |
|---|---|---|
| **1차** | `best_epoch >= 3` | ← **점수보다 이게 먼저** |
| 2차 | macro-F1 이 기준선 대비 **+0.02** 이상 (잡음 밖) | |

**둘 다 만족하는 판이 없으면 축을 닫습니다.** "0에폭이 진짜 최고" 도 결론이고,
그러면 원인은 학습률이 아니라 다른 데 있습니다.

## 판 4개

| | 헤드 lr | 백본 ×배수 | 백본 lr | warmup |
|---|---|---|---|---|
| **A 기준선** | 3e-4 | 0.30 | 9e-5 | 2 |
| B | 1e-4 | 0.30 | 3e-5 | 2 |
| C | 3e-4 | 0.10 | 3e-5 | 2 |
| D | 1e-4 | 0.10 | 1e-5 | 1 |

B 와 C 는 백본 lr 이 같습니다(3e-5) — **머리를 낮춘 것과 몸통을 낮춘 것**을
갈라 보려는 배치입니다. 둘이 같이 움직이면 백본 lr 이 원인이고, B 만 움직이면
헤드 쪽입니다.

## 붙일 것

| 입력 | 왜 |
|---|---|
| `m2.5` 크롭 (2단계용) | **`f320` 은 필요 없습니다** — 1단계를 안 돌립니다 |

⚠️ **STEP 16 과 같은 데이터(365,428행)여야 합니다.** 옛 45,885행 크롭을 붙이면
기준선이 달라져서 비교가 안 됩니다. 실험 이름에 학습셋 크기가 찍히니 확인하세요.


In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
# ★ 이 노트북이 사는 브랜치. **여기서 못 박지 않으면 "main" 을 받습니다.**
#    캐글/콜랩은 리포가 없는 상태로 시작해서 아래 _ROOT 탐색이 실패하고,
#    예전 기본값이 "main" 이었습니다. main 이 뒤처져 있으면 **셀은 최신인데
#    src/ 만 옛것**인 채로 돕니다 — 실제로 며칠 그랬습니다 (main 75445c0).
#    첫 셀은 그 상태에서도 "코드 버전 …" 을 태연히 찍습니다.
NB_BRANCH = "claude/dog-disease-diagnosis-model-1s6jtf"
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()

# ⚠️ "지금 리포 안인가" 를 **폴더 이름으로만** 보면 안 됩니다. 주피터에서
#    notebooks/*.ipynb 를 열면 cwd 가 `.../deeplearning_test/notebooks` 라
#    이름이 안 맞고, 그러면 **리포 안에 리포를 또 clone** 합니다
#    (실제로 런팟에서 .../notebooks/deeplearning_test 가 생겼습니다).
#    위로 거슬러 올라가며 **진짜 리포 루트**를 찾습니다.
_p = os.path.abspath(_cwd)
_ROOT = None
while True:
    if (os.path.isdir(os.path.join(_p, ".git"))
            and os.path.isfile(os.path.join(_p, "src", "env.py"))):
        _ROOT = _p
        break
    _up = os.path.dirname(_p)
    if _up == _p:
        break
    _p = _up

if _ROOT:
    DIR = _ROOT           # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or NB_BRANCH

# ⚠️ 예전엔 fetch/reset 을 **둘 다 check=False** 로 불렀습니다. 실패해도 조용히
#    넘어가서, 캐글 클론이 **지워진 커밋(75445c0)에 붙박인 채 며칠을 돌았습니다.**
#    src/ 를 아무리 고쳐 푸시해도 안 실렸고, 첫 셀은 "코드 버전 …" 을 태연히
#    찍었습니다. 그 줄을 믿을 수 없다는 게 제일 나빴습니다.
#    → 이제 실패하면 **말하고, 클론을 지우고 다시 받습니다.**
#    (Kaggle Persistence 를 'Files' 로 켜두면 /kaggle/working 이 살아남아
#     낡은 클론이 계속 재사용됩니다 — 그 경우에도 여기서 복구됩니다.)
def _git(*args, cwd=None):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)


def _fresh_clone(dst, branch):
    import shutil as _sh
    _sh.rmtree(dst, ignore_errors=True)
    r = _git("clone", "-b", branch, "--depth", "1", REPO, dst)
    if r.returncode != 0:
        raise RuntimeError("git clone 실패:\n" + (r.stderr or "")[-800:])


_need_clone = not os.path.isdir(os.path.join(DIR, ".git"))
if not _need_clone:
    # shallow clone 이라 origin/<브랜치> 대신 FETCH_HEAD 로 맞춥니다
    # (히스토리가 갈리면 origin/<브랜치> 가 옛 커밋을 가리킨 채 남습니다)
    r = _git("-C", DIR, "fetch", "--depth", "1", "origin", BRANCH)
    if r.returncode != 0:
        print("⚠️ git fetch 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
        _need_clone = True
    else:
        r = _git("-C", DIR, "reset", "--hard", "FETCH_HEAD")
        if r.returncode != 0:
            print("⚠️ git reset 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
            _need_clone = True

if _need_clone:
    _fresh_clone(DIR, BRANCH)

# ★ 정말 최신인지 **확인**합니다. 위가 다 성공해도 여기서 한 번 더 봅니다 —
#   "최신이라고 믿었는데 아니었다" 가 이 프로젝트에서 가장 비쌌던 실패입니다.
_local = _git("-C", DIR, "rev-parse", "HEAD").stdout.strip()
_remote = _git("-C", DIR, "ls-remote", REPO, f"refs/heads/{BRANCH}").stdout.split()
_remote = _remote[0] if _remote else ""
if _remote and _local and not _remote.startswith(_local[:8]) and not _local.startswith(_remote[:8]):
    print("\n" + "!" * 66)
    print(f"🚨 코드가 최신이 아닙니다 — 로컬 {_local[:8]} / 원격 {_remote[:8]}")
    print("   클론을 지우고 다시 받습니다.")
    print("!" * 66 + "\n")
    _fresh_clone(DIR, BRANCH)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", _git("-C", DIR, "log", "--oneline", "-1").stdout.strip())
print("브랜치      :", BRANCH,
      f"(원격 {_remote[:8]})" if _remote else "(원격 확인 실패)")
if BRANCH != NB_BRANCH:
    print(f"⚠️ 이 노트북이 만들어진 브랜치({NB_BRANCH})가 아닙니다 —")
    print("   src/ 가 셀보다 뒤처져 있을 수 있습니다. 아래 [nb] 줄을 꼭 보세요.")

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ 임대 GPU 이미지(런팟 등)의 파이썬은 **externally managed** 입니다 (PEP 668).
#    그냥 설치하면 첫 시도가 통째로 거부돼서, 재시도 로직이 있어도 무서운
#    에러 덩어리가 먼저 찍힙니다. 처음부터 허용해두면 그 소음이 없습니다.
#    Colab/Kaggle 에는 이 제약이 없어서 이 변수는 무해합니다.
os.environ["PIP_BREAK_SYSTEM_PACKAGES"] = "1"
os.environ["UV_BREAK_SYSTEM_PACKAGES"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-09-04.2"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 0-b. 데이터 붙이기

In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ── 다른 환경에서 학습한 체크포인트 가져오기 (Colab → Kaggle 이주) ──────
#    Colab 에서 이미 학습을 끝냈다면, Drive 의 dogskin_work/checkpoints 를
#    Kaggle 데이터셋으로 올린 뒤 그 경로를 여기에 주세요.
#    가져온 실험은 '완료' 로 인식되어 학습 셀이 ⏭️ 로 건너뜁니다.
#
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
import torch
from src import labels, split, crop, data, models, train, evaluate, stages, experiments
from src.config import CLASSES, CFG, ADOPTED_STAGE2_CROP

env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# ── STEP 16 에서 확정된 설정 — 여기서 고르는 게 아닙니다 ──────────
BEST_CROP    = ADOPTED_STAGE2_CROP     # "m2.5"
IMG_SIZE     = 384
STAGE2_MODEL = "convnextv2_base"
AUG          = "default"               # 증강 축은 고정 — 한 번에 한 축만 움직입니다

# ── 스윕용 ────────────────────────────────────────────────────
SUBSET_FRAC  = 0.40    # 학습셋만 40%. 검증셋은 그대로
EPOCHS       = 12

# ★ STEP 16 풀 실행 실측 (val). 서브셋은 이보다 낮게 나오는 게 정상입니다 —
#   **판끼리만** 비교하세요. 근거: docs/results/STEP16_전체데이터_재학습_실측.md
STEP16 = {"macro_f1": 0.5999, "best_epoch": 0, "n_epochs": 14,
          "scale_drop": 0.268, "lr": 3e-4, "backbone_lr_mult": 0.3}

df = labels.load(env.work_root() / "manifests" / "manifest_final.parquet")
print(f"{len(df):,}행 / 개체 {df['animal_id'].nunique():,}마리")
if len(df) < 300_000:
    print("\n⚠️ 행이 365,428 보다 훨씬 적습니다 — **옛 데이터**일 수 있습니다.")
    print("   STEP 16 기준선과 비교가 안 됩니다. 붙인 데이터셋을 확인하세요.")

s2_all = stages.to_stage2(crop.switch_tag(df, BEST_CROP))
split.verify(s2_all, fold=0, strict=True)

_tr, _va = split.get_fold(s2_all, CFG().use_fold)
print(f"2단계 train {len(_tr):,} → 서브셋 {int(len(_tr) * SUBSET_FRAC):,} / "
      f"val {len(_va):,}")

## 1. 시간 먼저 — 캐글 세션 안에 들어가나

⚠️ 캐글은 세션 하나가 **9시간**이고 주당 30시간입니다. 4판이 안 들어가면
`SUBSET_FRAC` 이나 `EPOCHS` 를 줄이세요. 중간에 끊기면 `train.fit` 이
이어받긴 하지만, 세션이 바뀌면 체크포인트가 없어집니다.

In [ ]:
# 한 판이 얼마나 걸리는지 추정합니다 (실측 표 기반, src/experiments.py)
est = experiments.estimate_runtime(
    [(STAGE2_MODEL, EPOCHS)] * 4, img_size=IMG_SIZE,
    n_train=int(len(_tr) * SUBSET_FRAC), device=DEV)

print("\n⚠️ 추정치입니다 — 실측이 아닙니다 (작업 규칙 1).")
print("   9시간을 넘길 것 같으면 SUBSET_FRAC 을 0.25 로 줄이세요.")

## 2. 네 판

한 판이 끝날 때마다 표가 갱신됩니다. **`best` 열만 보세요** — 거기가
0~2 에 머물면 그 판도 같은 병입니다.

In [ ]:
# ⚠️ 네 판이 **같은 백본·같은 크롭·같은 증강**입니다. 학습률만 다릅니다.
#    experiments.train_and_measure 가 lr/배수/warmup 을 **실험 이름에 붙입니다** —
#    안 붙이면 네 판이 한 폴더를 공유해 뒤의 세 판이 조용히 건너뛰어집니다.
PLANS = [
    {"lr": 3e-4, "backbone_lr_mult": 0.3, "warmup_epochs": 2},   # A 기준선
    {"lr": 1e-4, "backbone_lr_mult": 0.3, "warmup_epochs": 2},   # B 헤드를 낮춤
    {"lr": 3e-4, "backbone_lr_mult": 0.1, "warmup_epochs": 2},   # C 백본을 낮춤
    {"lr": 1e-4, "backbone_lr_mult": 0.1, "warmup_epochs": 1},   # D 둘 다 + warmup 1
]

runs = []
for i, plan in enumerate(PLANS):
    print(f"\n{'█' * 70}\n 판 {'ABCD'[i]} — {plan}\n{'█' * 70}")
    runs.append(experiments.train_and_measure(
        s2_all, stage=2, img_size=IMG_SIZE, crop_tag=BEST_CROP, device=DEV,
        model_name=STAGE2_MODEL, finetune="moderate", aug=AUG,
        epochs=EPOCHS, subset_frac=SUBSET_FRAC, n_robust=2000, **plan))
    r = runs[-1]
    print(f"\n  → best epoch {r['best_epoch']} / {r['n_epochs']}에폭   "
          f"macro-F1 {r['score']:.4f}   실험 이름 {r['exp_name']}")

## 3. 판정

`lr_report()` 가 **위에서 못 박은 기준 그대로** 판정합니다. 노트북 셀이 아니라
`src/experiments.py` 에 있어서, 결과를 보고 기준을 바꿀 수 없습니다.

In [ ]:
verdict = experiments.lr_report(runs, baseline_lr=3e-4, baseline_mult=0.3)

In [ ]:
import json

W = env.work_root()
(W / "reports").mkdir(parents=True, exist_ok=True)

out = {
    "step": "STEP 17 — 2단계 학습률",
    "setup": {"model": STAGE2_MODEL, "crop": BEST_CROP, "img_size": IMG_SIZE,
              "aug": AUG, "epochs": EPOCHS, "subset_frac": SUBSET_FRAC,
              "n_train": runs[0]["n_train"] if runs else None},
    "step16_baseline": STEP16,
    "verdict": verdict,
    "runs": [{k: v for k, v in r.items() if k != "report"} for r in runs],
}
p = W / "reports" / "step17_lr.json"
p.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")

print("=" * 70)
print(" STEP 17 결과 (이 블록을 복사해서 공유하세요)")
print("=" * 70)
print(f"  서브셋 {SUBSET_FRAC:.0%} · {EPOCHS}에폭 · 학습 "
      f"{runs[0]['n_train']:,}장 · {STAGE2_MODEL} @ {IMG_SIZE}px\n")
print(f"  {'판':<3}{'헤드lr':>9}{'×배수':>7}{'백본lr':>9}"
      f"{'macro-F1':>10}{'best':>6}{'배율하락':>9}{'분':>6}")
for i, r in enumerate(runs):
    d = r.get("scale_drop")
    print(f"  {'ABCD'[i]:<3}{r['lr']:>9.0e}{r['backbone_lr_mult']:>7.2f}"
          f"{r['backbone_lr']:>9.0e}{r['score']:>10.4f}{r['best_epoch']:>6}"
          f"{(f'{d:.1%}' if d is not None else '—'):>9}{r['minutes']:>6.0f}")
print(f"\n  판정: {verdict.get('verdict')}")
print("=" * 70)
print(f"\n저장: {p}")
print("\n⚠️ 이 숫자는 **서브셋**입니다. STEP 16 의 0.5999 와 직접 비교하지 마세요.")

## 다음

| 판정 | 다음에 할 것 |
|---|---|
| **채택** | 06 을 그 학습률로 **전체 데이터** 재실행 → 새 기준선. A4·배율도 다시 잼 |
| **학습은 되나 점수는 그대로** | '0에폭 best' 는 고쳤으니 그 설정을 06 에 반영하되, 성능 개선은 다른 축에서 찾음 (데이터·증강) |
| **축 닫힘** | 학습률 아님. `docs/results/` 에 기록하고 A4 원인 분석(07)으로 넘어감 |

어느 쪽이든 **`docs/results/STEP17_2단계_학습률_실측.md` 에 남깁니다.**
"축이 닫혔다" 도 결론이고, 안 적으면 몇 주 뒤에 같은 걸 또 돌립니다.

⚠️ **채택이 나와도 서브셋 결과입니다.** STEP 12 의 `convnextv2_base` 도
서브셋에서 이겼고 전체에서 확인하는 데 STEP 16 까지 걸렸습니다.